<div align="right" style="text-align: right"><i>by Gemini Neon 4<br>Foreword by Peter Norvig<br>Nov 2026</i></div>

# Forward 

*I was jealous of all the recent AI-assisted proofs of important mathematical results, so I used AI  to solve a not-quite-as-important problem that had been open for 11 years and has never been solved before: the problem of finding a shortest possible portmantout. Recall that a "portmanteau" is a word that squishes together other words, like "mathlete = math + athlete." A "portmantout" squishes together **all** the words in a word list. Tom Murphy VII [introduced the problem](https://tom7.org/portmantout/) in 2015 with a 108.709-word list, and found an upper bound on the total length of the string; I was able to [improve on the bound](Portmantout.ipynb) in 2018. Now, at last, the problem is completely solved, thanks to Gemini Neon (and Claude Fable, which was [equally able to solve it](Portmantout-Claude.ipynb)). I believe that the models from just two months ago could not do this; I checked with Opus 5.2.*

*This forward is written by me, Peter Norvig; the rest of this notebook is by Gemini Neon.*

|Letters|Author|Method|% of optimal|
|--:|--|:--|--:|
|630,408|Tom Murphy|Greedy joining particles ([portmantout.cc](https://sourceforge.net/p/tom7misc/svn/HEAD/tree/trunk/portmantout/))|117%|
|554,113|Peter Norvig|Greedy path with bridges ([Portmantout.ipynb](https://github.com/norvig/pytudes/blob/main/ipynb/Portmantout.ipynb))|103%|
|543,607|Peter Norvig|Greedy path with near-subwords ([Portmantout.ipynb](https://github.com/norvig/pytudes/blob/main/ipynb/Portmantout.ipynb))|101%|
|**536,186**|Gemini Neon|Directed Rural Postman (this notebook)|**100%**|
|**536,186**|Claude Fable|Directed Rural Postman [Portmantout-Claude.ipynb](Portmantout-Claude.ipynb)|**100%**|


–*Peter Norvig*

<hr>

# Provably Shortest Portmantout Words

This notebook analyzes Peter Norvig's [`Portmantout.ipynb`](https://github.com/norvig/pytudes/blob/main/ipynb/Portmantout.ipynb) and implements a pure-SciPy **exact algorithm** that finds a **provably shortest possible portmantout** for Tom Murphy VII's 108,709-word `wordlist.asc` (or any  wordset $W$).

## 1. Analysis of Norvig's Portmantout.ipynb

Given a finite wordset $W$, a **portmantout** is a string $S$ produced by a sequence of overlapping words $P = [\text{Step}(0, w_0), \text{Step}(k_1, w_1), \dots, \text{Step}(k_{m-1}, w_{m-1})]$ where:
1. Every word $w \in W$ appears as a substring of $S$.
2. Every consecutive pair overlaps by $1 \le k_i < \min(|w_{i-1}|, |w_i|)$ characters ($w_{i-1}[-k_i:] = w_i[:k_i]$).
3. $S = w_0 + w_1[k_1:] + \dots + w_{m-1}[k_{m-1}:]$, so $|S| = |w_0| + \sum_{i=1}^{m-1} (|w_i| - k_i)$.

In `Portmantout.ipynb`, Norvig's greedy `natalie` heuristic partitions `wordlist.asc` into 64,389 nonsubwords and 44,320 subwords, precomputes 70,300 minimal-excess 1- and 2-step bridges from 11,536 short words, and greedily extends a single path—achieving **553,472 letters** on `W` and **543,369 letters** on near-subword-merged `W1` (vs. Tom Murphy VII's ~630,000). Greedy falls short because of severe start/end letter imbalances (45.1% of nonsubwords end in `s`, `y` has a 1:39 start-to-end ratio, and no nonsubwords end in `j` or `q`), myopic local choices, a restricted short-word bridge pool, and the lack of a global lower bound.


## 2. Exact Algorithm: Overlap-Gadget Directed Rural Postman Problem (DRPP)

Instead of treating **words as nodes** (a Traveling Salesperson Problem), we treat **overlaps as nodes** and **words as directed 3-arc gadgets**:

1. **Nonsubword equivalence**: Any valid portmantout string $S$ covering $W$ can be parsed as a valid path visiting every nonsubword $w \in W_{\text{req}}$ at least once without changing $|S|$.
2. **Overlap-gadget graph**: Let $A = \text{Prefixes}(W) \cap \text{Suffixes}(W)$. Create nodes `START`, `END`, one node per overlap $a \in A$, and $(w_{\text{in}}, w_{\text{out}})$ per usable word $w \in W$:
   - Entry arcs $p \to w_{\text{in}}$ with cost $-|p|$, bounds $[0, \infty)$ for each proper prefix $p \in A$ of $w$.
   - Middle arc $w_{\text{in}} \to w_{\text{out}}$ with cost $|w|$, bounds $[1, \infty)$ if $w \in W_{\text{req}}$ else $[0, \infty)$.
   - Exit arcs $w_{\text{out}} \to s$ with cost $0$, bounds $[0, \infty)$ for each proper suffix $s \in A$ of $w$.
   - `START` $\to w_{\text{in}}$ and $w_{\text{out}} \to$ `END` (cost $0$) for each $w \in W_{\text{req}}$, plus return arc `END` $\to$ `START` with fixed flow $[1, 1]$.
3. **Three-stage exact solver**:
   - **Stage 1 (Min-Cost Circulation LP)**: Solve $\min c^\top x$ s.t. $Bx = 0, l \le x \le u$ via `scipy.optimize.linprog(..., method="highs")`. Total unimodularity of $B$ guarantees an integral optimal circulation and a provable lower bound $LB = c^\top x^*$, plus exact dual node potentials.
   - **Stage 2 (Zero-Cost Potential-Guided Reconnection)**: If the active subgraph has stray components, attempt to merge them into the `START` component along zero-reduced-cost residual cycles, preserving $OPT = LB$ (on `wordlist.asc`, the lone stray `okeydoke` links in at zero cost for $OPT = 536{,}186$).
   - **Stage 3 (Exact Subtour Elimination MILP)**: If any stray component $S \not\ni \text{START}$ cannot merge at zero cost, add the cut $\sum_{e \in \delta^+(S)} x_e \ge 1$ and solve with `scipy.optimize.milp` until connected—guaranteeing a **provably shortest** portmantout even when $OPT > LB$.
   - **Stage 4 (Eulerian Walk Extraction)**: Extract the walk via Hierholzer's algorithm and verify every constraint with `is_portman`.


In [1]:
from __future__ import annotations

import os
import time
import urllib.request
from collections import Counter, defaultdict, namedtuple
from dataclasses import dataclass
from typing import Iterable

import numpy as np
from scipy import sparse
from scipy.optimize import Bounds, LinearConstraint, linprog, milp


## 3. Core Data Types, Verifier, and Overlap-Gadget Graph


In [2]:
Step = namedtuple("Step", ("overlap", "word"))


def portman(P: list[Step]) -> str:
    """Compute the portmantout string S from the path P."""
    return "".join(step.word[step.overlap:] for step in P)


def subparts(word: str) -> set[str]:
    """All non-empty proper substrings of word."""
    n = len(word)
    return {
        word[i:j]
        for i in range(n)
        for j in range(i + 1, n + (1 if i != 0 else 0))
    }


def prefixes(word: str) -> list[str]:
    """All non-empty proper prefixes of word."""
    return [word[:i] for i in range(1, len(word))]


def suffixes(word: str) -> list[str]:
    """All non-empty proper suffixes of word, longest first."""
    return [word[i:] for i in range(1, len(word))]


def split_subwords(words: Iterable[str]) -> tuple[set[str], set[str]]:
    """Partition words into (nonsubwords, subwords)."""
    W = set(words)
    subwords = {part for w in W for part in subparts(w) if part in W}
    nonsubwords = W - subwords
    return nonsubwords, subwords


def is_portman(P: list[Step], W: set[str], nonsubwords: set[str] | None = None) -> bool:
    """Verify P is a valid portmantout path covering every word in W."""
    if not P:
        return len(W) == 0
    if P[0].overlap != 0 or P[0].word not in W:
        return False
    for i in range(1, len(P)):
        ov, w = P[i]
        prev = P[i - 1].word
        if w not in W:
            return False
        if not (1 <= ov < min(len(prev), len(w))):
            return False
        if prev[-ov:] != w[:ov]:
            return False
    req = nonsubwords if nonsubwords is not None else split_subwords(W)[0]
    used = {step.word for step in P}
    return req <= used <= W


@dataclass
class OptimalResult:
    path: list[Step]
    string: str
    length: int
    lp_lower_bound: int
    lp_components: int
    reconnected_zero_cost: bool
    milp_cuts_added: int
    solve_seconds: float
    num_nodes: int
    num_arcs: int
    num_overlaps: int
    num_nonsubwords: int
    num_usable_subwords: int


class OverlapGraph:
    """Overlap-gadget directed graph for exact portmantout optimization."""

    def __init__(self, W: set[str]):
        self.W = set(W)
        self.nonsubwords, self.subwords = split_subwords(self.W)

        all_prefixes = {p for w in self.W for p in prefixes(w)}
        all_suffixes = {s for w in self.W for s in suffixes(w)}
        self.overlaps = sorted(all_prefixes & all_suffixes)
        overlap_set = set(self.overlaps)

        self.usable_subwords = sorted(
            w for w in self.subwords
            if any(p in overlap_set for p in prefixes(w))
            and any(s in overlap_set for s in suffixes(w))
        )
        self.nonsub_list = sorted(self.nonsubwords)
        self.usable_words = self.nonsub_list + self.usable_subwords

        self.START = 0
        self.END = 1
        next_id = 2

        self.overlap_node: dict[str, int] = {}
        self.node_overlap: dict[int, str] = {}
        for a in self.overlaps:
            self.overlap_node[a] = next_id
            self.node_overlap[next_id] = a
            next_id += 1

        self.word_in: dict[str, int] = {}
        self.word_out: dict[str, int] = {}
        self.in_node_word: dict[int, str] = {}
        for w in self.usable_words:
            self.word_in[w] = next_id
            self.in_node_word[next_id] = w
            self.word_out[w] = next_id + 1
            next_id += 2

        self.num_nodes = next_id

        u_list, v_list, c_list, lb_list, ub_list = [], [], [], [], []

        def add_arc(u: int, v: int, c: float, lo: float, hi: float):
            u_list.append(u)
            v_list.append(v)
            c_list.append(c)
            lb_list.append(lo)
            ub_list.append(hi)

        # Arc 0: END -> START with fixed flow 1
        self.return_arc_idx = 0
        add_arc(self.END, self.START, 0.0, 1.0, 1.0)

        nonsub_set = self.nonsubwords
        for w in self.usable_words:
            win, wout = self.word_in[w], self.word_out[w]
            is_req = w in nonsub_set

            add_arc(win, wout, float(len(w)), 1.0 if is_req else 0.0, np.inf)

            for p in prefixes(w):
                if p in overlap_set:
                    add_arc(self.overlap_node[p], win, float(-len(p)), 0.0, np.inf)

            for s in suffixes(w):
                if s in overlap_set:
                    add_arc(wout, self.overlap_node[s], 0.0, 0.0, np.inf)

            if is_req:
                add_arc(self.START, win, 0.0, 0.0, np.inf)
                add_arc(wout, self.END, 0.0, 0.0, np.inf)

        self.num_arcs = len(u_list)
        self.u_arr = np.array(u_list, dtype=np.int32)
        self.v_arr = np.array(v_list, dtype=np.int32)
        self.cost_arr = np.array(c_list, dtype=np.float64)
        self.lb_arr = np.array(lb_list, dtype=np.float64)
        self.ub_arr = np.array(ub_list, dtype=np.float64)

        rows = np.concatenate([self.v_arr, self.u_arr])
        cols = np.concatenate([np.arange(self.num_arcs), np.arange(self.num_arcs)])
        data = np.concatenate([np.ones(self.num_arcs), -np.ones(self.num_arcs)])
        self.B = sparse.csc_matrix((data, (rows, cols)), shape=(self.num_nodes, self.num_arcs))

    def active_components(self, flow: np.ndarray) -> list[set[int]]:
        """Weakly connected components of {e : flow[e] > 0}, with START component first."""
        adj: dict[int, list[int]] = defaultdict(list)
        active_nodes: set[int] = set()
        for e in np.nonzero(flow > 0)[0]:
            u, v = int(self.u_arr[e]), int(self.v_arr[e])
            adj[u].append(v)
            adj[v].append(u)
            active_nodes.add(u)
            active_nodes.add(v)

        visited: set[int] = set()
        comps: list[set[int]] = []
        for node in active_nodes:
            if node in visited:
                continue
            comp: set[int] = set()
            stack = [node]
            visited.add(node)
            while stack:
                curr = stack.pop()
                comp.add(curr)
                for nxt in adj[curr]:
                    if nxt not in visited:
                        visited.add(nxt)
                        stack.append(nxt)
            comps.append(comp)
        comps.sort(key=lambda c: (0 if self.START in c else 1, -len(c)))
        return comps

    def try_zero_cost_connect(self, flow: np.ndarray, duals: np.ndarray) -> tuple[np.ndarray, bool]:
        """Stage 2: Merge disconnected components at zero excess cost using LP dual potentials."""
        flow = flow.copy()
        rc_plus = self.cost_arr + (duals[self.v_arr] - duals[self.u_arr])
        rc_minus = self.cost_arr - (duals[self.v_arr] - duals[self.u_arr])
        free_mask = (self.ub_arr == np.inf) & (flow == 0)
        rc = rc_plus if (np.any(free_mask) and rc_plus[free_mask].min() >= rc_minus[free_mask].min()) else rc_minus

        zero_rc = np.abs(rc) <= 1e-5

        while True:
            comps = self.active_components(flow)
            if len(comps) <= 1:
                return flow, True

            main, stray = comps[0], comps[1]
            s0, m0 = next(iter(stray)), next(iter(main))

            res_adj: dict[int, list[tuple[int, int, int]]] = defaultdict(list)
            for e in np.nonzero(zero_rc & (flow < self.ub_arr))[0]:
                res_adj[int(self.u_arr[e])].append((int(self.v_arr[e]), int(e), +1))
            for e in np.nonzero(zero_rc & (flow > self.lb_arr))[0]:
                res_adj[int(self.v_arr[e])].append((int(self.u_arr[e]), int(e), -1))

            def bfs_path(src: int, dst: int) -> list[tuple[int, int]] | None:
                parent: dict[int, tuple[int, int, int] | None] = {src: None}
                q = [src]
                head = 0
                while head < len(q):
                    curr = q[head]
                    head += 1
                    for nxt, e, d in res_adj.get(curr, ()):
                        if nxt == dst:
                            path = [(e, d)]
                            p = curr
                            while parent[p] is not None:
                                pn, pe, pd = parent[p]
                                path.append((pe, pd))
                                p = pn
                            path.reverse()
                            return path
                        if nxt not in parent:
                            parent[nxt] = (curr, e, d)
                            q.append(nxt)
                return None

            p1 = bfs_path(s0, m0)
            p2 = bfs_path(m0, s0) if p1 is not None else None
            if p1 is None or p2 is None:
                return flow, False

            delta: dict[int, int] = defaultdict(int)
            for e, d in p1 + p2:
                delta[e] += d

            if not all(self.lb_arr[e] <= flow[e] + d <= self.ub_arr[e] for e, d in delta.items()):
                return flow, False

            new_flow = flow.copy()
            for e, d in delta.items():
                new_flow[e] += d

            if len(self.active_components(new_flow)) >= len(comps):
                return flow, False
            flow = new_flow

    def extract_path(self, flow: np.ndarray) -> list[Step]:
        """Stage 4: Extract an Eulerian trail from START to END via Hierholzer's algorithm."""
        adj: dict[int, list[int]] = defaultdict(list)
        for e in np.nonzero(flow > 0)[0]:
            if e == self.return_arc_idx:
                continue
            u, v = int(self.u_arr[e]), int(self.v_arr[e])
            adj[u].extend([v] * int(flow[e]))

        stack = [self.START]
        circuit: list[int] = []
        while stack:
            v = stack[-1]
            if adj[v]:
                stack.append(adj[v].pop())
            else:
                circuit.append(stack.pop())
        circuit.reverse()
        assert circuit[0] == self.START and circuit[-1] == self.END

        steps: list[Step] = []
        for i in range(1, len(circuit)):
            node = circuit[i]
            if node in self.in_node_word:
                word = self.in_node_word[node]
                prev = circuit[i - 1]
                ov = 0 if prev == self.START else len(self.node_overlap[prev])
                steps.append(Step(ov, word))
        return steps


## 4. Exact Solver (`solve_optimal_portmantout`)


In [3]:
def solve_optimal_portmantout(W: Iterable[str]) -> OptimalResult:
    """Find a provably shortest portmantout for wordset W using SciPy HiGHS."""
    t0 = time.perf_counter()
    W_set = set(W)
    nonsub, _ = split_subwords(W_set)
    if len(nonsub) == 0:
        return OptimalResult([], "", 0, 0, 0, False, 0, time.perf_counter() - t0, 0, 0, 0, 0, 0)
    if len(nonsub) == 1:
        w = next(iter(nonsub))
        return OptimalResult([Step(0, w)], w, len(w), len(w), 1, False, 0, time.perf_counter() - t0, 0, 0, 0, 1, 0)

    G = OverlapGraph(W_set)

    # Stage 1: Exact Min-Cost Circulation LP (HiGHS)
    res = linprog(
        c=G.cost_arr,
        A_eq=G.B,
        b_eq=np.zeros(G.num_nodes),
        bounds=np.column_stack([G.lb_arr, G.ub_arr]),
        method="highs",
    )
    if not res.success:
        raise ValueError(f"No valid portmantout exists for W (LP status: {res.message})")
    assert np.allclose(res.x, np.round(res.x), atol=1e-5), "LP solution was unexpectedly fractional"

    flow = np.round(res.x).astype(np.int64)
    lp_lb = int(round(res.fun))
    lp_components = len(G.active_components(flow))
    reconnected_zero_cost = False
    milp_cuts_added = 0

    # Stage 2: Zero-Cost Potential-Guided Residual Reconnection
    if lp_components > 1:
        flow2, connected = G.try_zero_cost_connect(flow, res.eqlin.marginals)
        if connected:
            flow = flow2
            reconnected_zero_cost = True

    # Stage 3: Exact Subtour Elimination via Branch-and-Cut MILP (only if OPT > LB)
    if len(G.active_components(flow)) > 1:
        cut_rows: list[np.ndarray] = []
        integrality = np.ones(G.num_arcs, dtype=int)
        bounds = Bounds(lb=G.lb_arr, ub=G.ub_arr)
        eq_constraint = LinearConstraint(G.B, lb=np.zeros(G.num_nodes), ub=np.zeros(G.num_nodes))

        while True:
            comps = G.active_components(flow)
            if len(comps) <= 1:
                break
            for S in comps[1:]:
                in_S_u = np.isin(G.u_arr, list(S))
                in_S_v = np.isin(G.v_arr, list(S))
                cut_rows.append((in_S_u & (~in_S_v)).astype(np.float64))
                milp_cuts_added += 1

            C_mat = sparse.csc_matrix(np.vstack(cut_rows))
            cut_constraint = LinearConstraint(C_mat, lb=np.ones(len(cut_rows)), ub=np.full(len(cut_rows), np.inf))
            mres = milp(
                c=G.cost_arr,
                integrality=integrality,
                bounds=bounds,
                constraints=[eq_constraint, cut_constraint],
            )
            if not mres.success:
                raise ValueError(f"No connected portmantout exists (MILP status: {mres.message})")
            flow = np.round(mres.x).astype(np.int64)

    # Stage 4: Eulerian Trail Extraction & Verification
    path = G.extract_path(flow)
    S = portman(path)
    opt_len = int(round(float(G.cost_arr @ flow)))
    assert len(S) == opt_len, f"Length mismatch: {len(S)} != {opt_len}"
    assert is_portman(path, W_set, G.nonsubwords), "Output failed is_portman verification!"

    return OptimalResult(
        path=path,
        string=S,
        length=len(S),
        lp_lower_bound=lp_lb,
        lp_components=lp_components,
        reconnected_zero_cost=reconnected_zero_cost,
        milp_cuts_added=milp_cuts_added,
        solve_seconds=time.perf_counter() - t0,
        num_nodes=G.num_nodes,
        num_arcs=G.num_arcs,
        num_overlaps=len(G.overlaps),
        num_nonsubwords=len(G.nonsubwords),
        num_usable_subwords=len(G.usable_subwords),
    )


## 5. Download `wordlist.asc` and Compute the Provably Shortest Portmantout

We download Tom Murphy VII's 108,709-word list directly from `https://norvig.com/ngrams/wordlist.asc` and run `solve_optimal_portmantout(W)`.


In [4]:
WORDLIST_URL = "https://norvig.com/ngrams/wordlist.asc"
WORDLIST_PATH = "wordlist.asc"

if not os.path.exists(WORDLIST_PATH):
    urllib.request.urlretrieve(WORDLIST_URL, WORDLIST_PATH)

W = set(open(WORDLIST_PATH).read().split())
nonsub, sub = split_subwords(W)
total_letters = sum(len(w) for w in W)

print(f"Loaded {len(W):,d} words ({len(nonsub):,d} nonsubwords, {len(sub):,d} subwords, {total_letters:,d} total letters)")

%time opt_res = solve_optimal_portmantout(W)

Loaded 108,709 words (64,389 nonsubwords, 44,320 subwords, 931,823 total letters)
CPU times: user 3min 29s, sys: 1.65 s, total: 3min 31s
Wall time: 3min 31s


In [5]:
print("\n=== Provably Optimal Portmantout ===")
print(f"Optimal length       : {opt_res.length:,d} letters (compression ratio {total_letters / opt_res.length:.3f}x)")
print(f"LP lower bound       : {opt_res.lp_lower_bound:,d} letters (gap: {opt_res.length - opt_res.lp_lower_bound})")
print(f"LP components        : {opt_res.lp_components} (zero-cost merge: {opt_res.reconnected_zero_cost}, MILP cuts: {opt_res.milp_cuts_added})")
print(f"Path steps           : {len(opt_res.path):,d} ({len(opt_res.path) - opt_res.num_nonsubwords:,d} bridge/repeat steps)")
print(f"Graph size           : {opt_res.num_nodes:,d} nodes, {opt_res.num_arcs:,d} arcs, {opt_res.num_overlaps:,d} overlaps")
print(f"Solve time           : {opt_res.solve_seconds:.2f}s")
print(f"\nFirst 120 chars:\n{opt_res.string[:120]}...")
print(f"Last  120 chars:\n...{opt_res.string[-120:]}")
print("\nFirst 24 steps:")
print(" ".join(f"{w[:ov]}·{w[ov:]}" for ov, w in opt_res.path[:24]))


=== Provably Optimal Portmantout ===
Optimal length       : 536,186 letters (compression ratio 1.738x)
LP lower bound       : 536,186 letters (gap: 0)
LP components        : 2 (zero-cost merge: True, MILP cuts: 0)
Path steps           : 99,739 (35,350 bridge/repeat steps)
Graph size           : 228,709 nodes, 904,140 arcs, 11,293 overlaps
Solve time           : 211.13s

First 120 chars:
jealousiesynodulesynodulartworksynoddynexusesynoddlesynoddingbatwomenovocainextricablyrejudgementwistsynoddersynoddedwel...
Last  120 chars:
...edaubedamnsaberedactedabbledabbedamnederailedabbingabardinesabbatsabbaticalsabbathsabotswanaesthetizationonideologically

First 24 steps:
·jealousies s·ynod nod·ules s·ynod nod·ular ar·tworks s·ynod nod·dy dy·ne ne·xuses s·ynod nod·dles s·ynod nod·ding ding·bat bat·women n·ovocaine ine·xtricably ly·re re·judge judge·ment ent·wists s·ynod nod·ders


## 6. Top Reused Bridges & Worst Letter Imbalances

A summary showing the most frequently reused bridge words and the most imbalanced start/end letters.


In [6]:
word_counts = Counter(step.word for step in opt_res.path)
print("Top 10 most-reused bridge words:")
for w, count in word_counts.most_common(10):
    if count > 1:
        kind = "subword" if w in sub else "nonsubword"
        print(f"  {w+":":5s} {count:5,d} uses ({kind})")

starts = Counter(w[0] for w in nonsub)
ends = Counter(w[-1] for w in nonsub)
imbalance = sorted(
    ((ends[ch] - starts[ch], ch, starts[ch], ends[ch]) for ch in set(starts) | set(ends)),
    key=lambda x: abs(x[0]),
    reverse=True,
)
print("\nTop 8 most imbalanced start/end letters (nonsubwords):")
print(f"  {'Letter':>6} {'Starts':>8} {'Ends':>8} {'Ends-Starts':>12}")
for diff, ch, s_cnt, e_cnt in imbalance[:8]:
    print(f"  {ch:>6} {s_cnt:8,d} {e_cnt:8,d} {diff:+12,d}")

Top 10 most-reused bridge words:
  so:   3,680 uses (subword)
  of:   2,292 uses (subword)
  gun:  2,083 uses (subword)
  sam:  1,971 uses (subword)
  lyre: 1,799 uses (subword)
  sub:  1,773 uses (subword)
  doc:  1,626 uses (subword)
  sac:  1,418 uses (subword)
  sin:  1,350 uses (subword)
  gob:  1,052 uses (subword)

Top 8 most imbalanced start/end letters (nonsubwords):
  Letter   Starts     Ends  Ends-Starts
       s    7,388   29,056      +21,668
       y      207    8,086       +7,879
       c    5,849      908       -4,941
       p    4,977      123       -4,854
       g    2,177    6,343       +4,166
       b    3,776        6       -3,770
       d    4,093    7,520       +3,427
       a    3,528      384       -3,144
